# Chapitre 9 - Cours 2 : arbres de régression et de classification (CART)

Ce notebook suit la section 9.2 du livre. Les arbres **partitionnent l'espace des
entrées en rectangles** et ajustent un modèle très simple (une constante) dans chacun.
Nous allons :

1. décrire les partitions binaires récursives (figure 9.2) ;
2. faire croître un arbre de **régression** (critère des moindres carrés, éq. (9.13)) ;
3. l'**élaguer** par le critère coût-complexité (éq. (9.16)) ;
4. passer à la **classification** : mesures d'impureté de Gini, d'entropie et d'erreur
   (figure 9.3) ;
5. discuter les autres questions pratiques (variables qualitatives, pertes, instabilité) ;
6. reproduire l'exemple **spam** : validation croisée (figure 9.4), arbre élagué
   (figure 9.5), table 9.3 et courbes ROC (figure 9.6).

Tout est implémenté à la main dans la classe `ArbreCART` du module `esl/ch09_arbres.py`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch09_arbres import (ArbreCART, cart_cv, impurity, trace_arbre, roc_curve,
                             auc, GAM, SPAM_NAMES)

set_style()
rng = np.random.RandomState(0)

## 1. Partitions binaires récursives (figure 9.2)

Considérons deux entrées $X_1, X_2\in[0,1]$. Une partition quelconque du carré par des
segments parallèles aux axes (panneau de gauche ci-dessous) produit des régions
difficiles à décrire. On se restreint donc aux partitions **binaires récursives** : on
coupe d'abord l'espace en deux selon une règle $X_j\le s$, puis on recoupe chacune des
deux moitiés, et ainsi de suite (panneau de droite).